# Executive Summary: Fraud Detection Proof of Concept

This notebook demonstrates a machine learning solution designed to identify potentially fraudulent credit card transactions. The goal is to support business stakeholders in reducing financial loss while maintaining a positive customer experience.

### What this solution does
- Analyzes historical transaction data
- Identifies unusual patterns (anomalies)
- Flags high-risk transactions for further review

### Why it matters
Fraud detection is a balance between:
- **Catching fraud (true positives)**
- **Avoiding disruption to legitimate customers (false positives)**

This model provides a scalable, automated way to assist fraud teams in prioritizing investigations and reducing operational costs.

---

## Pipeline Diagram

```
[Raw Transaction Data]
          ↓
[Data Cleaning & Preparation]
          ↓
[Model Training (Anomaly Detection)]
          ↓
[Fraud Scoring]
          ↓
[Evaluation & Business Insights]
```

---

## Azure ML Components and Roles

| Azure ML Component | Role in Process |
|--------------------|----------------|
| Workspace | Central environment to manage experiments and models |
| Compute Instance | Executes notebook and training jobs |
| Datastore | Stores input transaction data |
| Dataset | Structured representation of transaction data |
| Experiment | Tracks model runs and results |
| Model | Trained anomaly detection algorithm |
| Endpoint (future) | Deploys model for real-time scoring |




---

### Step 1: Import the Packages and Connect to Azure

In [ ]:
# Step 1: Import Packages and Connect to your Azure Workspace
from azureml.core import Workspace, Dataset         # see https://pypi.org/project/azureml-core/
import pandas as pd                                 # see https://pandas.pydata.org/docs/
from sklearn.ensemble import IsolationForest        # see https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.IsolationForest.html
from sklearn.metrics import classification_report   # see https://scikit-learn.org/stable/modules/generated/sklearn.metrics.classification_report.html
from azureml.core.model import Model                # see https://docs.microsoft.com/en-us/python/api/azureml-core/azureml.core.model?view=azure-ml-py

### Step 2: Data Loading

**What is happening:**  
We are bringing in historical transaction data into the system.

In [ ]:
# You only need to run this if you've imported this notebook to Azure AI Machine Learning Studio - Notebook,
# in which case you'll also need to upload the config.json file to the same directory as this notebook,
# and then execute this code to determine the current working directory.
import os
print("Current working directory:", os.getcwd())
print("Files in this directory:", os.listdir())


In [ ]:
# if you're running locally then use this ...
path = None

# alternatively, if you're running in Azure AI Machine Learning Studio - Notebook, then use this ...
# (make sure to upload the config.json file to the same directory as this notebook)
#  and then execute this code to determine the current working directory.
path='Users/[REPLACE-THIS-WITH-YOUR-USERNAME]/config.json'
ws = Workspace.from_config(path=path)
dataset = Dataset.get_by_name(ws, name='creditcard_fraud')
df = dataset.to_pandas_dataframe()
df.head()

### Step 3: Prepare the Data

**What is happening:**  
We prepare the transaction data so the model can evaluate all the features on the same scale.

**Why it matters:**  
Well-prepared data improves the model’s accuracy and ensures reliable results during training and evaluation.

In [ ]:
df['Amount'] = (df['Amount'] - df['Amount'].mean()) / df['Amount'].std()
X = df.drop(columns=['Class', 'Time'])
y = df['Class']

### Step 4: Train the model

**What is happening:**  
We train an Isolation Forest model to identify unusual transactions. The model works by separating (“isolating”) data points — transactions that are very different from the rest are flagged as potential anomalies.

**Why it matters:**  
Fraudulent transactions are rare and often look different from normal activity. This approach is effective at spotting those rare, unusual patterns quickly, even in large volumes of data.

**Executive takeaway:**  
Enables fast and scalable detection of suspicious transactions, helping identify potential fraud early while handling large datasets efficiently.

In [ ]:
model = IsolationForest(contamination=0.0017, random_state=42)
model.fit(X)
y_pred = model.predict(X)
y_pred = [1 if x == -1 else 0 for x in y_pred]

### Step 5: Evaluating the Anomaly Detection Model

**What is happening:**  
We assess how well the model detects fraud using performance metrics derived from a confusion matrix. These metrics show how accurately the model identifies both normal and fraudulent transactions.

**Why it matters:**  
While the model appears highly accurate overall, this is misleading because fraud is rare. The key insight is that the model performs well on normal transactions but misses most fraudulent ones and has a low success rate when flagging fraud.

**Executive takeaway:**
Despite high overall accuracy, the model is not yet reliable for fraud detection — it misses a significant portion of fraud cases and requires further improvement before business deployment.


In [ ]:
# Step 5: Evaluate Model
print(classification_report(y, y_pred))

### Step 6 (Optional): Register the Model


In [ ]:
import joblib                                       # see https://joblib.readthedocs.io/en/latest/
                                                    #     Joblib is a set of tools to provide lightweight pipelining in Python
joblib.dump(model, 'isolation_forest.pkl')
Model.register(model_path='isolation_forest.pkl',
               model_name='creditcard_if_model',
               workspace=ws)


### Step 7: Visualize a Count of Predicted Anomalies

**What is happening:**  
We visualize how many transactions the model classified as normal versus potentially fraudulent, providing a quick snapshot of its predictions.

**Why it matters:**  
This helps us understand whether the model is too aggressive (flagging too many transactions) or too conservative (missing fraud), which directly impacts operational workload and fraud risk.

**Executive takeaway:**  
Provides a quick health check of model behavior, ensuring the balance between catching fraud and minimizing unnecessary alerts is appropriate for business needs.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Add predictions to the original dataframe
df['predicted_anomaly'] = y_pred

# Count of predicted anomalies
sns.countplot(x='predicted_anomaly', data=df)
plt.title('Count of Predicted Anomalies')
plt.xlabel('Anomaly (1) vs Normal (0)')
plt.ylabel('Count')
plt.show()


### Step 7 (continued): Visualize Transaction Amount by Prediction Class

**What is happening:**    
The boxplot below compares the **amount of money** in transactions that the model predicted as **normal (0)** or **anomalous/fraud (1)**.
- You may see that predicted frauds (`1`) tend to have **more extreme** or **variable amounts**.
- This could suggest that the model is flagging **unusually high or low transaction amounts** as suspicious.

**Why it matters:**  
This chart helps you:
- Understand what kinds of amounts the model thinks are suspicious.
- Spot any bias in the model (e.g. only flagging large transactions).
- Decide whether you need to normalize, transform, or engineer features differently.

In [ ]:
plt.figure(figsize=(10, 6))
sns.boxplot(data=df, x='predicted_anomaly', y='Amount')
plt.title('Transaction Amount by Prediction Class')
plt.show()


### Step 7 (continued): SHAP Beeswarm Plot – Feature Importance for Anomaly Detection

**What is happening:**  
We use SHAP to explain which factors most influenced the model’s decisions and how each feature contributed to identifying potential fraud.

**Why it matters:**  
This provides transparency into how the model works, helping stakeholders understand and trust its decisions — especially important in fraud detection where accountability matters.

**Executive takeaway:**  
Makes the model’s decisions explainable, increasing trust and enabling better oversight, validation, and future improvements.

In [ ]:
import shap

explainer = shap.Explainer(model, X)
shap_values = explainer(X[:100])
shap.plots.beeswarm(shap_values)


# Business Impact Assessment

---

## Cost-Benefit Analysis

- **False Positives (legitimate transactions flagged):**
  - Customer frustration
  - Increased support workload

- **Missed Fraud:**
  - Direct financial losses
  - Reputational damage

**Recommendation:**  
Tune the model threshold based on acceptable business risk.

---

## Recommendations for Improvement

- Add richer behavioral features
- Test multiple model types
- Implement continuous retraining
- Move toward real-time fraud detection

---

## Risk Assessment & Mitigation

| Risk | Mitigation |
|------|-----------|
| Model drift | Scheduled retraining |
| Data bias | Regular audits |
| Over-flagging | Threshold tuning |
| Under-detection | Ensemble models |

---

## Stakeholder Communication Plan

- Position model as a **decision-support tool**
- Share regular performance reports
- Clearly explain limitations and error trade-offs
- Align with fraud, CX, and compliance teams